# 01 Verificacion PINN FUCCI

Reanálisis de los resultados del póster con archivos locales del repositorio. Ejecutar después de instalar requirements.txt. No entrena PINNs. Las salidas nuevas se guardan en results/generated/; las tablas publicadas se conservan.


In [ ]:
OUTPUT_NAME = 'resultados_01_verificacion'
from pathlib import Path
import json, hashlib, sys
import os
for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(key, '1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, brentq
from scipy.stats import chi2, probplot
from IPython.display import display
try:
    from threadpoolctl import threadpool_limits
    thread_limit = threadpool_limits(limits=1)
except ImportError:
    thread_limit = None  # Instalar requirements.txt para controlar hilos en una sesión ya abierta.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src/common.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src/common.py').exists():
    raise RuntimeError('Abra el notebook desde la raíz del repositorio o notebooks/.')
sys.path.insert(0, str(ROOT / 'src'))
from common import load_data, data_arrays, FDSolver, SIGMA, TIMES, BOUNDS
INPUT = ROOT / 'data'
OUT = ROOT / 'results/generated' / OUTPUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size':11, 'figure.dpi':110})
inputs = [INPUT/'datos_reorganizados.csv',
          ROOT/'results/published/real_s1.csv', ROOT/'results/published/real_s2.csv']
manifest = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}
import scipy, platform
(OUT/'configuracion.json').write_text(json.dumps(dict(sha256=manifest, sigma=0.05,
    nx=139, inicial='coseno K=8', dominio=[0,1242], python=platform.python_version(),
    numpy=np.__version__, scipy=scipy.__version__), indent=2), encoding='utf-8')


## Modelo compartido con la implementación local
139 nodos; dominio 0–1242 µm; condición inicial coseno K=8; flujo nulo. Se mantienen las cotas locales: difusión 1–2000, tasas 0,0001–0,2. Estas son las cotas de nuestro código, no una reproducción literal de todas las cotas del artículo.

In [ ]:
# El solver se importa de src/common.py para evitar duplicaciones.


## Datos y ajuste clásico desde varios inicios
Se excluye t=0 del objetivo (144 residuos). Se registra éxito de cada optimización.

In [ ]:
df = load_data(INPUT/'datos_reorganizados.csv')
assert len(df)==96 and not df.duplicated(['x','t']).any()
assert np.isfinite(df.to_numpy()).all()
xs, ts, R, G = data_arrays(df)
assert np.allclose(xs, np.arange(24)*54) and np.array_equal(ts,TIMES)
fd = FDSolver(xs,R[0],G[0])
NAMES = {1:['D','kr','kg'],2:['Dr','Dg','kr','kg']}
def unpack(theta,sc):
    return (theta[0],theta[0],theta[1],theta[2]) if sc==1 else tuple(theta)
def scale_for(sc): return np.array([1000 if n.startswith('D') else .05 for n in NAMES[sc]])
def bounds_for(sc):
    return np.array([BOUNDS[n][0] for n in NAMES[sc]]),np.array([BOUNDS[n][1] for n in NAMES[sc]])
def residual(theta,sc,solver=None):
    ro,go=(solver or fd).at_obs(*unpack(theta,sc))
    return np.r_[(ro[1:]-R[1:]).ravel(),(go[1:]-G[1:]).ravel()]/SIGMA

def fit(sc,solver=None):
    scale=scale_for(sc); lo,hi=bounds_for(sc)
    starts=([[700,.03,.07],[300,.05,.1],[1500,.02,.05]] if sc==1 else
            [[600,900,.03,.07],[1100,100,.028,.073],[10,1900,.031,.069],[1500,500,.05,.1]])
    fits=[least_squares(lambda q:residual(q*scale,sc,solver),np.array(p)/scale,
        bounds=(lo/scale,hi/scale),xtol=1e-9,ftol=1e-9,gtol=1e-8,max_nfev=500) for p in starts]
    valid=[a for a in fits if a.success and np.isfinite(a.cost)]
    if not valid: raise RuntimeError('Ningún ajuste convergió: revisar antes de seguir.')
    best=min(valid,key=lambda a:a.cost)
    return best.x*scale,float(2*best.cost),pd.DataFrame([dict(inicio=i,exito=a.success,chi2=2*a.cost,nfev=a.nfev) for i,a in enumerate(fits)])
fits={}
for sc in (1,2):
    theta,cost,log=fit(sc); fits[sc]=(theta,cost)
    log.to_csv(OUT/f'inicios_mle_s{sc}.csv',index=False)
    pd.DataFrame({'parametro':NAMES[sc],'estimacion':theta}).to_csv(OUT/f'mle_s{sc}.csv',index=False)
    print('Escenario',sc,dict(zip(NAMES[sc],theta)),'chi²=',cost); display(log)


## 1 · Verificar cada vector PINN con el solver clásico
El CSV conserva el error del campo de la red. Aquí se recalcula el error de la EDP con sus parámetros. Sin los pesos `.pt` no se reconstruye el campo PINN ni se mide directamente la diferencia entre campos. Un parámetro fuera de las cotas MLE se evalúa, pero se marca.

In [ ]:
tables=[]
for sc in (1,2):
    d=pd.read_csv(ROOT/'results/published'/f'real_s{sc}.csv'); names=NAMES[sc]; lo,hi=bounds_for(sc)
    assert len(d)>0 and not d.rep.duplicated().any()
    assert np.isfinite(d[['pinn_'+n for n in names]+['pinn_chi2']].to_numpy()).all()
    rows=[]
    for _,row in d.iterrows():
        p=row[['pinn_'+n for n in names]].to_numpy(float)
        e=residual(p,sc); c=float(e@e)
        rows.append(dict(escenario=sc,rep=int(row.rep),chi2_red=row.pinn_chi2,
            chi2_fd=c,delta_fd_red=c-row.pinn_chi2,delta_fd_mle=c-fits[sc][1],
            rmse_fd=np.sqrt(np.mean((SIGMA*e)**2)),fuera_cotas=bool(np.any((p<lo)|(p>hi))),
            **dict(zip(names,p))))
    t=pd.DataFrame(rows); tables.append(t); display(t)
    t.to_csv(OUT/f'verificacion_s{sc}.csv',index=False)
    if (t.delta_fd_mle < -1e-4).any():
        print('ATENCIÓN: hay una solución FD mejor que el MLE; revisar cotas e inicios antes de interpretar.')
    fig,ax=plt.subplots(figsize=(6,4))
    ax.scatter(t.chi2_red,t.chi2_fd); limits=[min(t.chi2_red.min(),t.chi2_fd.min()),max(t.chi2_red.max(),t.chi2_fd.max())]
    ax.plot(limits,limits,'k--'); ax.set(xlabel='Error χ² del campo PINN',ylabel='Error χ² de FD con θ PINN',title=f'Escenario {sc}')
    fig.tight_layout();fig.savefig(OUT/f'validacion_s{sc}.png',dpi=300);plt.show()
t=tables[1]
fig,ax=plt.subplots(figsize=(6,5)); dots=ax.scatter(t.Dr,t.Dg,c=t.chi2_fd,cmap='viridis',s=70)
ax.scatter(*fits[2][0][:2],marker='*',s=180,c='red',label='MLE-FD')
ax.set(xlabel='Dr (µm²/h)',ylabel='Dg (µm²/h)',title='Difusiones por entrenamiento');ax.legend()
fig.colorbar(dots,ax=ax,label='χ² recalculado por FD');fig.tight_layout();fig.savefig(OUT/'difusiones_s2.png',dpi=300);plt.show()


## 2 · Ajustes y residuos del MLE
Los residuos se definen como observado − predicho. Las figuras son diagnósticos dentro de muestra, no validación predictiva. El Q–Q no prueba independencia espacial. Comparar la dispersión observada con σ=0,05 sin reinterpretarla automáticamente como error experimental.

In [ ]:
for sc in (1,2):
    ro,go=fd.at_obs(*unpack(fits[sc][0],sc)); rows=[]
    fig,axes=plt.subplots(2,2,figsize=(10,7),sharey=True)
    for j,ax in enumerate(axes.flat):
        for y,p,color,label in [(R,ro,'#b83240','G1'),(G,go,'#16805a','S/G2/M')]:
            ax.scatter(xs,y[j],s=18,color=color);ax.plot(xs,p[j],color=color,label=label)
            if j>0:
                rows.extend(dict(x=x,t=ts[j],poblacion=label,observado=a,predicho=b,residuo=a-b) for x,a,b in zip(xs,y[j],p[j]))
        ax.set(title=f'{ts[j]:g} h',xlabel='x (µm)',ylabel='Densidad');ax.legend()
    fig.tight_layout();fig.savefig(OUT/f'ajuste_mle_s{sc}.png',dpi=300);plt.show()
    res=pd.DataFrame(rows);res.to_csv(OUT/f'residuos_s{sc}.csv',index=False)
    display(res.groupby(['poblacion','t']).residuo.agg(['mean','std','count']))
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    for (pop,tim),g in res.groupby(['poblacion','t']):
        axes[0].plot(g.x,g.residuo,'.-',label=f'{pop}, {tim:g} h')
    axes[0].axhline(0,color='k',lw=.8);axes[0].set(xlabel='x (µm)',ylabel='Observado − predicho');axes[0].legend(fontsize=7)
    for pop,g in res.groupby('poblacion'): axes[1].scatter(g.predicho,g.residuo,label=pop,s=16)
    axes[1].axhline(0,color='k',lw=.8);axes[1].set(xlabel='Predicho',ylabel='Residuo');axes[1].legend()
    probplot(res.residuo/SIGMA,dist='norm',plot=axes[2]);axes[2].set_title('Q–Q residuos / σ supuesto')
    fig.tight_layout();fig.savefig(OUT/f'diagnosticos_s{sc}.png',dpi=300);plt.show()


## 3 · Sensibilidad a la condición inicial (sin reentrenar)
Compara el coseno original con interpolación lineal no negativa sobre el mismo dominio y malla. Cambian suavizado y positividad a la vez: no atribuir diferencias exclusivamente a los valores negativos. Estos ajustes alternativos no reemplazan la referencia principal ni validan la PINN bajo otra condición inicial.

In [ ]:
print('Mínimos de la condición inicial:',fd.r0.min(),fd.g0.min())
alt=FDSolver(xs,R[0],G[0],ic='interp')
rows=[]
for sc in (1,2):
    p,c,log=fit(sc,alt)
    for name,a,b in zip(NAMES[sc],fits[sc][0],p):
        rows.append(dict(escenario=sc,parametro=name,coseno=a,interpolacion=b,cambio_relativo=(b-a)/a,chi2_alternativo=c))
t=pd.DataFrame(rows);display(t);t.to_csv(OUT/'sensibilidad_inicial.csv',index=False)
